In [1]:
import torch
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)

from peft import LoraConfig
from trl import SFTTrainer

In [2]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: NVIDIA GeForce RTX 5060 Ti


In [3]:
train_dataset = load_dataset(
    "json",
    data_files="../data/qwen_finetune_train.jsonl",
    split="train"
)

val_dataset = load_dataset(
    "json",
    data_files="../data/qwen_finetune_val.jsonl",
    split="train"
)

print(train_dataset)
print(val_dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 49
})
Dataset({
    features: ['messages'],
    num_rows: 12
})


In [4]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Qwen model loaded.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen model loaded.


In [5]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print(peft_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'q_proj', 'v_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [6]:
training_args = TrainingArguments(
    output_dir="../models/qwen_oss_compliance_lora",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    learning_rate=2e-4,

    num_train_epochs=3,

    logging_steps=1,

    save_strategy="epoch",

    eval_strategy="epoch",

    fp16=True,

    report_to="none"
)

In [7]:
trainer = SFTTrainer(
    model=model,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    peft_config=peft_config,

    args=training_args
)

Tokenizing train dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

In [8]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,0.938405,0.923328
2,0.437310,0.491942
3,0.543698,0.444159


TrainOutput(global_step=147, training_loss=1.1331866967434785, metrics={'train_runtime': 16.4205, 'train_samples_per_second': 8.952, 'train_steps_per_second': 8.952, 'total_flos': 130492545048576.0, 'train_loss': 1.1331866967434785})

In [9]:
trainer.save_model("../models/qwen_oss_compliance_lora")

print("Fine-tuned model saved successfully.")

Fine-tuned model saved successfully.
